# FlowSense (Teeuwen & Gil 2025) - what is in each file

Zenodo 10.5281/zenodo.16794871. Three folders: `traffic_flows`, `road_network`, `ground_truth`.
For every file this notebook shows the table (rows, columns, first rows) and marks which columns the
dashboard uses (**USED** column in the column tables).

Run the cells top to bottom. The large Goteborg flows file (293 MB) takes about a minute to load once.

In [ ]:
import os, glob
import pandas as pd
import pyogrio

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 40)

BASE = r"C:\Users\saraabo\flowsense-trafficflows"
TF = os.path.join(BASE, "traffic_flows", "traffic_flows")
RN = os.path.join(BASE, "road_network", "road_network")
GT = os.path.join(BASE, "ground_truth", "ground_truth")

_cache = {}
def load(path):
    """read a geojson once (geometry kept as a column called geometry)"""
    if path not in _cache:
        _cache[path] = pyogrio.read_dataframe(path)
    return _cache[path]

def show_file(path, used=None, rows=5):
    """summary line, column table with USED flag and example value, then the first rows"""
    used = used or {}
    df = load(path)
    print(os.path.basename(path))
    print(f"  {len(df):,} rows | {df.shape[1]-1} columns (+ geometry) | crs {df.crs} | "
          f"{df.geometry.geom_type.value_counts().to_dict()}")
    d = df.drop(columns="geometry")
    info = pd.DataFrame({
        "dtype": d.dtypes.astype(str),
        "non_null": d.notna().sum(),
        "example": [d[c].dropna().iloc[0] if d[c].notna().any() else None for c in d.columns],
        "USED": [used.get(c, "") for c in d.columns],
    })
    display(info)
    display(d.head(rows))

## 1. traffic_flows - the phone data

One Goteborg file (the one the dashboard uses) and 20 Stockholm files (not used).

In [ ]:
rows = []
for f in sorted(glob.glob(os.path.join(TF, "*.geojson"))):
    i = pyogrio.read_info(f)
    rows.append({"file": os.path.basename(f), "MB": round(os.path.getsize(f) / 1e6),
                 "rows": i["features"], "columns": len(i["fields"]),
                 "city": "Goteborg (USED)" if "_gbg_" in f else "Stockholm (not used)"})
pd.DataFrame(rows)

### 1a. Goteborg flows - `flows_gbg_flows_2024_random1to2.geojson`

One row per road piece **per direction**. Used: `maxspeed`, the nine `trajcount_minavgspeed*` columns and
the geometry. Everything else (`relflow_*`, `minnpoints`, `maxavgdist`, the IDs) is not used.

In [ ]:
GBG_FLOWS = os.path.join(TF, "flows_gbg_flows_2024_random1to2.geojson")
used = {"maxspeed": "YES - speed class"}
for t in ["0", "2.5", "5", "7.5", "10", "12.5", "15", "17.5", "20"]:
    used[f"trajcount_minavgspeed{t}"] = "YES - nine speed filters"
used["geometry"] = "YES"
flows = load(GBG_FLOWS)
show_file(GBG_FLOWS, used)

In [ ]:
# how the 56 count/share columns group, and how many roads each one reaches
d = flows.drop(columns="geometry")
cnt = [c for c in d.columns if c.startswith("trajcount_")]
summary = pd.DataFrame({
    "family": [c.split("_")[1].rstrip("0123456789.") for c in cnt],
    "column": cnt,
    "roads_with_count": [(pd.to_numeric(d[c]) > 0).sum() for c in cnt],
    "total_crossings": [pd.to_numeric(d[c]).sum() for c in cnt],
    "max_on_one_road": [pd.to_numeric(d[c]).max() for c in cnt],
})
summary

### 1b. Stockholm flows (not used)

Each of the 20 files has 5 columns, except `random1to11` which has the same 59 as Goteborg.
Shown here only so you can see them.

In [ ]:
sth = sorted(glob.glob(os.path.join(TF, "*sthlm*.geojson")))
small = [f for f in sth if "1to11" not in f][0]
print("columns in a small Stockholm file:", list(pyogrio.read_info(small)["fields"]))
display(load(small).drop(columns="geometry").head())

## 2. road_network - the road map the counts sit on

Used: the Goteborg **edges** file, columns `u`, `v`, `key`, `length` and the geometry.
It lines up row by row with the flows file. The nodes file, the Stockholm files and the `raw` folder are not used.

In [ ]:
rows = []
for f in sorted(glob.glob(os.path.join(RN, "**", "*.*"), recursive=True)):
    if os.path.basename(f).startswith("."):
        continue
    ours = ("edges_gbg" in f)
    rows.append({"file": os.path.relpath(f, RN), "MB": round(os.path.getsize(f) / 1e6, 1),
                 "used": "YES" if ours else ""})
pd.DataFrame(rows)

In [ ]:
EDGES = os.path.join(RN, "preprocessed", "road_network_trafikverket_edges_gbg.geojson")
used = {"u": "YES - merge directions", "v": "YES - merge directions", "key": "YES - merge directions",
        "length": "YES - centrality weight", "geometry": "YES - road line",
        "highway": "carried, but = 'unclassified' for every row"}
edges = load(EDGES)
show_file(EDGES, used)

In [ ]:
# the road class column carries no information
edges.highway.astype(str).value_counts()

In [ ]:
NODES = os.path.join(RN, "preprocessed", "road_network_trafikverket_nodes_gbg.geojson")
show_file(NODES, {})   # not used: the junction IDs already sit in u and v

In [ ]:
# the two files align row by row - this is how a count is joined to its road piece
print("same number of rows:", len(flows) == len(edges))
print("first RLID in each:", flows.RLID.iloc[0], edges.RLID.iloc[0])
print("maxspeed matches in:", (flows.maxspeed.astype(str) == edges.maxspeed.astype(str)).mean().round(3), "of rows")

### 2b. The raw folder (not used)

In [ ]:
isa = os.path.join(RN, "raw", "ISA.gpkg")
print(pyogrio.list_layers(isa))
print("ISA.gpkg is 1.3 GB - first 5 features only:")
display(pyogrio.read_dataframe(isa, max_features=5).drop(columns="geometry"))

In [ ]:
for t in ["Kolumnnamn.txt", "Licensinformation.txt"]:
    print("-----", t)
    print(open(os.path.join(RN, "raw", t), encoding="utf-8", errors="replace").read()[:1500])

## 3. ground_truth - real traffic counts used to check the phone data

Two Goteborg files are used. The Stockholm file is only used for an hourly shape (a feature the dashboard no longer shows).

In [ ]:
HW = os.path.join(GT, "preprocessed", "ground_truth_flows_cars_gbg_highway_2023.geojson")
used = {"adt_samtliga_fordon": "YES - the count", "element_id": "YES - site name",
        "direction": "YES", "extent_length": "read", "matmetod": "read (how the number was obtained)",
        "geometry": "YES"}
hw = load(HW)
show_file(HW, used)

In [ ]:
# how reliable the Trafikverket numbers are: matmetod = measurement method
t = hw.groupby("matmetod").adt_samtliga_fordon.agg(["count", "min", "median", "max"])
t["english (my translation)"] = t.index.map({"Stickprovsmätning": "sample measurement", "Bedömt flöde med stödmätning": "assessed flow, with a supporting measurement", "Bedömt flöde utan stödmätning": "assessed flow, without a supporting measurement"})
t

In [ ]:
LOC = os.path.join(GT, "preprocessed", "ground_truth_flows_cars_gbg_local_2023.geojson")
used = {"ådt": "YES - the count", "mätplatsnamn": "YES - site name", "riktning": "YES", "geometry": "YES"}
show_file(LOC, used)

In [ ]:
STH = os.path.join(GT, "preprocessed", "ground_truth_flows_cars_sthlm_2024.geojson")
sth_gt = load(STH)
print(f"{len(sth_gt)} rows, {sth_gt.shape[1]} columns - Stockholm 2024, not used for count sites")
cols = [c for c in sth_gt.columns if c.startswith("trafikflöde_") and "alla_fordon" in c and "dygn" in c]
display(sth_gt[["namn", "hastighet_medel"] + cols[:6]].head())

In [ ]:
# the two count files we use, side by side on the same footing
a = hw[["adt_samtliga_fordon"]].rename(columns={"adt_samtliga_fordon": "adt"}).assign(source="Trafikverket highway link")
b = load(LOC)[["ådt"]].rename(columns={"ådt": "adt"}).assign(source="Goteborg municipal count")
both = pd.concat([a, b])
both = both[both.adt > 0]
both.groupby("source").adt.describe()

---
**Reading guide.** A column marked `YES` is read by the pipeline and ends up in `gothenburg-phone-js/data/sweep.json`.
`read` means the script loads the column but it does not change what the dashboard shows. Blank means unused.